# PAPER-EXP-06D — Frozen-Backbone Soft-Routed Residual Multidomain Adaptation

**Implementation v2 FP32 feature-extraction fix. Scientific protocol unchanged from the predeclared v1 design.**

Development only. Run unchanged with Internet OFF and the exact locked PAPEREXP06A Recovery + LITEGEN06A split inputs. Frozen tests and DERM12345 remain unopened.


In [1]:
from __future__ import annotations

import contextlib
import hashlib
import io
import json
import math
import random
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# ============================================================================
# LOCKED EXPERIMENT IDENTITY
# ============================================================================

EXPERIMENT = "PAPER-EXP-06D"
TITLE = "Frozen-Backbone Soft-Routed Residual Multidomain Adaptation"
CODE_VERSION = "PAPEREXP06D_FROZEN_06A_SOFT_ROUTED_RESIDUAL_2026-08-30_v2_FP32_FEATURE_EXTRACTION_FIX"

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUT = WORKING_ROOT / "PAPEREXP06D_Frozen_Backbone_Soft_Routed_Residual_Adaptation"
FIG = OUT / "figures"
TAB = OUT / "tables"
PRED = OUT / "validation_predictions"
CKPT = OUT / "checkpoints"
AUDIT = OUT / "audit"
for p in (OUT, FIG, TAB, PRED, CKPT, AUDIT):
    p.mkdir(parents=True, exist_ok=True)

CLASSES = ("MEL", "NV", "BCC", "AKIEC", "BKL")
C2I = {c: i for i, c in enumerate(CLASSES)}
DOMAINS = ("ISIC2018_HAM10000", "BCN20000", "Derm7pt", "PAD_UFES_20")
D2I = {d: i for i, d in enumerate(DOMAINS)}
PRESENT = {
    "ISIC2018_HAM10000": ("MEL", "NV", "BCC", "AKIEC", "BKL"),
    "BCN20000": ("MEL", "NV", "BCC", "AKIEC", "BKL"),
    "Derm7pt": ("MEL", "NV", "BCC", "BKL"),
    "PAD_UFES_20": ("MEL", "NV", "BCC", "AKIEC", "BKL"),
}
TOKENS = {
    "ISIC2018_HAM10000": ("skin-cancer-mnist-ham10000", "ham10000"),
    "BCN20000": ("bcn20000", "bcn-20000"),
    "Derm7pt": ("derm7pt", "derm-7pt", "7pt"),
    "PAD_UFES_20": ("pad-ufes", "pad_ufes", "padufes"),
}
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

TRAIN_NAME = "litegen06a_generalization_train.csv"
VALID_NAME = "litegen06a_generalization_validation.csv"
TRAIN_SHA = "2109861cfd88d8dceb8186d7abcda8849b011f0f283ba4843b8bcdaf6dc8868d"
VALID_SHA = "ce56623674b0c09e3602206e86de0d0d62e878b70cf15eaa5746139eeedabe5e"
BASELINE_VALIDATION_NAME = "FINAL_fixed_50B0_50B3_tta7_validation.csv"
BASELINE_VALIDATION_SHA = "013573675e58fdf930637aa47c435d2d0f029e41ad0d332089ce276b3e782714"

BASELINE_CKPT_HASHES = {
    "b0_seed_20260820": "51eb6d967c6805ad622863c9b2d586954a119c2330a6608e9a73c95b808e6932",
    "b0_seed_20260821": "a32e5e6679af64db64d7154d59dcf62d7f8ef230b3f029d0c610168ea49998c3",
    "b0_seed_20260822": "610a79ffda7719598d6b26009dfe1c5534a4c7951179f9b018a5381e04bf7f6d",
    "b3_seed_20260820": "a5d882e39255417a7171a5403b4298430d8ac0c3af7e31dd589538c73e950b19",
    "b3_seed_20260821": "8827e01a59205723c5fefad9449740129072edb07bd4870c1361d904e512bbdb",
    "b3_seed_20260822": "33b60778f13785ca7748d85a32421bb53a9044f23ed716aafb1c625238463e98",
}
B0_NAMES = tuple(k for k in BASELINE_CKPT_HASHES if k.startswith("b0_"))
B3_NAMES = tuple(k for k in BASELINE_CKPT_HASHES if k.startswith("b3_"))

# Baseline values are re-computed from the exact 06A TTA7 validation CSV and
# must agree with these locked values before training starts.
BASELINE_METRICS = {
    "equal_domain_macro_f1": 0.753800130522593,
    "equal_domain_balanced_accuracy": 0.7661758345771839,
    "equal_domain_melanoma_recall": 0.6949035812672177,
    "equal_domain_accuracy": 0.7988462056089941,
    "per_domain": {
        "ISIC2018_HAM10000": {
            "accuracy": 0.8539215686274509,
            "macro_f1": 0.7615328252488938,
        },
        "BCN20000": {
            "accuracy": 0.6954335986594051,
            "macro_f1": 0.641995416691071,
        },
        "Derm7pt": {
            "accuracy": 0.8450704225352113,
            "macro_f1": 0.8386466408268733,
        },
        "PAD_UFES_20": {
            "accuracy": 0.8009592326139089,
            "macro_f1": 0.7730256393235344,
        },
    },
}

# ============================================================================
# PREDECLARED 06D DESIGN -- NO SEARCH
# ============================================================================

IMAGE_SIZE = 300
FEATURE_BATCH_B0 = 32
FEATURE_BATCH_B3 = 16
FEATURE_WORKERS = 4
ADAPTER_BATCH = 512
ADAPTER_EPOCHS = 12
ADAPTER_SEEDS = (20260829, 20260830, 20260831)
ADAPTER_LR = 3.0e-4
ADAPTER_WEIGHT_DECAY = 1.0e-4
LABEL_SMOOTHING = 0.02
SOURCE_ANCHOR_WEIGHT = 1.0
ROUTER_AUX_WEIGHT = 0.10
RESIDUAL_ENERGY_WEIGHT = 0.01
RESIDUAL_LOGIT_CAP = 0.50
DROPOUT = 0.10
ROUTER_TEMPERATURE = 1.0

# System-level selection eligibility. Epoch 0 is the exact 06A baseline.
ELIGIBILITY = {
    "equal_domain_balanced_accuracy_min_delta": -0.0025,
    "equal_domain_melanoma_recall_min_delta": -0.0050,
    "ISIC2018_HAM10000_macro_f1_min_delta": -0.0050,
    "ISIC2018_HAM10000_accuracy_min_delta": -0.0050,
    "BCN20000_macro_f1_min_delta": -0.0025,
    "Derm7pt_macro_f1_min_delta": -0.0100,
    "PAD_UFES_20_macro_f1_min_delta": -0.0050,
    "PAD_UFES_20_accuracy_min_delta": -0.0050,
}

# Final replacement gate. A complex routed residual system must earn a
# meaningful improvement; tiny validation noise is not sufficient.
ACCEPTANCE = {
    "equal_domain_macro_f1_min_improvement": 0.0050,
    "equal_domain_balanced_accuracy_min_delta": -0.0025,
    "equal_domain_melanoma_recall_min_delta": -0.0050,
    "ISIC2018_HAM10000_macro_f1_min_delta": -0.0050,
    "ISIC2018_HAM10000_accuracy_min_delta": -0.0050,
    "BCN20000_macro_f1_min_delta": 0.0000,
    "Derm7pt_macro_f1_min_delta": -0.0100,
    "PAD_UFES_20_macro_f1_min_delta": -0.0050,
    "PAD_UFES_20_accuracy_min_delta": -0.0050,
}

# The split-freeze package legitimately contains these two inert metadata files.
# They are permitted to EXIST but are never opened by this experiment.
ALLOWED_DERM12345_METADATA_BASENAMES = {
    "derm12345_domain_holdout.csv",
    "litegen06a_derm12345_domain_holdout.csv",
}

# ============================================================================
# UTILITIES
# ============================================================================

def sha_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def sha_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


@contextlib.contextmanager
def amp(device: torch.device):
    if device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            yield
    else:
        yield


def json_dump(obj, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")


def assert_derm12345_boundary() -> dict:
    inert = []
    offenders = []
    for p in INPUT_ROOT.rglob("*"):
        text = str(p).lower()
        if "derm12345" not in text:
            continue
        if p.is_file() and p.name.lower() in ALLOWED_DERM12345_METADATA_BASENAMES:
            inert.append(str(p))
        else:
            offenders.append(str(p))
    if offenders:
        raise RuntimeError(
            "DERM12345 must remain outside PAPER-EXP-06D development. "
            f"Found prohibited DERM12345 attachment path(s); first={offenders[0]}"
        )
    print(f"[DERM12345 BOUNDARY] inert frozen holdout manifest(s) present but unused: {len(inert)}")
    return {"inert_metadata_paths_present_but_unread": inert, "prohibited_paths": offenders}


def exact_bytes(name: str, expected_sha: str) -> tuple[bytes, str]:
    for p in INPUT_ROOT.rglob(name):
        if p.is_file():
            b = p.read_bytes()
            if sha_bytes(b) == expected_sha:
                return b, str(p)
    for zp in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zp) as z:
                matches = [n for n in z.namelist() if Path(n).name == name]
                for member in matches:
                    b = z.read(member)
                    if sha_bytes(b) == expected_sha:
                        return b, f"{zp}::{member}"
        except zipfile.BadZipFile:
            pass
    raise FileNotFoundError(f"Could not find exact locked file {name} with SHA256 {expected_sha}")


def load_manifests() -> tuple[pd.DataFrame, pd.DataFrame, dict, dict]:
    trb, trsrc = exact_bytes(TRAIN_NAME, TRAIN_SHA)
    vab, vasrc = exact_bytes(VALID_NAME, VALID_SHA)
    tr = pd.read_csv(io.BytesIO(trb))
    va = pd.read_csv(io.BytesIO(vab))
    if len(tr) != 16220 or len(va) != 3895:
        raise RuntimeError("Frozen development row counts changed.")
    required = {"dataset", "image_hint", "image_id_norm", "group_id", "class_name", "label", "role"}
    for name, df in (("TRAIN", tr), ("VALIDATION", va)):
        missing = required - set(df.columns)
        if missing:
            raise RuntimeError(f"{name} missing columns: {sorted(missing)}")
        if set(df.dataset) != set(DOMAINS):
            raise RuntimeError(f"{name} domain set changed: {sorted(set(df.dataset))}")
        if not set(df.class_name).issubset(CLASSES):
            raise RuntimeError(f"{name} class set changed")
    leakage = {}
    for d in DOMAINS:
        a = tr[tr.dataset.eq(d)]
        b = va[va.dataset.eq(d)]
        leakage[d] = {
            "image_overlap": len(set(a.image_id_norm) & set(b.image_id_norm)),
            "group_overlap": len(set(a.group_id.astype(str)) & set(b.group_id.astype(str))),
        }
        if any(leakage[d].values()):
            raise RuntimeError(f"Development leakage detected in {d}: {leakage[d]}")
    sources = {"train": trsrc, "validation": vasrc}
    return tr.reset_index(drop=True), va.reset_index(drop=True), leakage, sources


def index_images(domain: str):
    stem = defaultdict(list)
    name = defaultdict(list)
    count = 0
    for p in INPUT_ROOT.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in EXTS:
            continue
        if not any(token in str(p).lower() for token in TOKENS[domain]):
            continue
        stem[p.stem.lower()].append(p)
        name[p.name.lower()].append(p)
        count += 1
    if count == 0:
        raise FileNotFoundError(f"No images found for {domain}")
    print("[INDEX]", domain, count)
    return stem, name


def dedup_candidates(candidates):
    unique = []
    seen = set()
    for p in candidates:
        if str(p) not in seen:
            seen.add(str(p))
            unique.append(p)
    if len(unique) <= 1:
        return unique
    hashes = {sha_file(p) for p in unique}
    if len(hashes) == 1:
        return [sorted(unique, key=lambda x: str(x).lower())[0]]
    return unique


def resolve_domain(domain: str, frame: pd.DataFrame) -> pd.DataFrame:
    stem, name = index_images(domain)
    out = frame.copy()
    paths = []
    for _, row in out.iterrows():
        hint = str(row.image_hint).strip()
        image_id = str(row.image_id_norm).strip().lower()
        basename = Path(hint.replace("\\", "/")).name.lower()
        candidates = dedup_candidates(name.get(basename, []) + stem.get(image_id, []))
        if domain == "Derm7pt" and len(candidates) > 1:
            normalized_hint = hint.replace("\\", "/").lower()
            suffix_matches = [
                p for p in candidates
                if str(p).replace("\\", "/").lower().endswith(normalized_hint)
            ]
            if len(suffix_matches) == 1:
                candidates = suffix_matches
        if len(candidates) != 1:
            raise RuntimeError(
                f"{domain} unresolved image {image_id}, hint={hint!r}, candidates={candidates[:6]}"
            )
        paths.append(str(candidates[0]))
    out["image_path"] = paths
    if out.image_path.duplicated().any():
        raise RuntimeError(f"{domain} duplicate resolved image paths")
    return out


def resolve_all(train: pd.DataFrame, valid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    trains = []
    valids = []
    for d in DOMAINS:
        both = pd.concat(
            [
                train[train.dataset.eq(d)].assign(_split="train"),
                valid[valid.dataset.eq(d)].assign(_split="validation"),
            ],
            ignore_index=True,
        )
        both = resolve_domain(d, both)
        trains.append(both[both._split.eq("train")].drop(columns="_split"))
        valids.append(both[both._split.eq("validation")].drop(columns="_split"))
    tr = pd.concat(trains, ignore_index=True)
    va = pd.concat(valids, ignore_index=True)
    tr_order = {(r.dataset, str(r.image_id_norm)): i for i, r in train.iterrows()}
    va_order = {(r.dataset, str(r.image_id_norm)): i for i, r in valid.iterrows()}
    tr["_order"] = [tr_order[(r.dataset, str(r.image_id_norm))] for _, r in tr.iterrows()]
    va["_order"] = [va_order[(r.dataset, str(r.image_id_norm))] for _, r in va.iterrows()]
    tr = tr.sort_values("_order").drop(columns="_order").reset_index(drop=True)
    va = va.sort_values("_order").drop(columns="_order").reset_index(drop=True)
    return tr, va


def find_06a_checkpoints() -> tuple[dict[str, Path], dict]:
    reverse = {v: k for k, v in BASELINE_CKPT_HASHES.items()}
    found: dict[str, Path] = {}
    extraction = OUT / "verified_06a_checkpoints"
    extraction.mkdir(exist_ok=True)
    locations = {}

    for p in INPUT_ROOT.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".pth", ".pt", ".ckpt"}:
            try:
                h = sha_file(p)
            except Exception:
                continue
            name = reverse.get(h)
            if name and name not in found:
                found[name] = p
                locations[name] = str(p)

    if set(found) != set(BASELINE_CKPT_HASHES):
        for zp in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zp) as z:
                    for member in z.infolist():
                        if not member.filename.lower().endswith((".pth", ".pt", ".ckpt")):
                            continue
                        b = z.read(member)
                        h = sha_bytes(b)
                        name = reverse.get(h)
                        if name and name not in found:
                            p = extraction / f"{name}_{h[:12]}.pth"
                            p.write_bytes(b)
                            found[name] = p
                            locations[name] = f"{zp}::{member.filename}"
            except zipfile.BadZipFile:
                pass
            if set(found) == set(BASELINE_CKPT_HASHES):
                break

    missing = sorted(set(BASELINE_CKPT_HASHES) - set(found))
    if missing:
        raise FileNotFoundError(
            "Attach exact PAPER-EXP-06A Recovery archive. Missing locked checkpoints: " + ", ".join(missing)
        )
    observed = {name: sha_file(path) for name, path in found.items()}
    if observed != BASELINE_CKPT_HASHES:
        raise RuntimeError("06A checkpoint hash verification failed")
    return found, locations


def load_baseline_validation(valid: pd.DataFrame) -> tuple[np.ndarray, dict, str]:
    b, source = exact_bytes(BASELINE_VALIDATION_NAME, BASELINE_VALIDATION_SHA)
    frame = pd.read_csv(io.BytesIO(b))
    required = {"dataset", "image_id_norm", "group_id", "class_name", "label"} | {
        f"prob_{c}" for c in CLASSES
    }
    missing = required - set(frame.columns)
    if missing:
        raise RuntimeError(f"06A baseline validation predictions missing columns: {sorted(missing)}")
    if len(frame) != len(valid):
        raise RuntimeError("06A baseline validation row count mismatch")
    keys_frame = list(zip(frame.dataset.astype(str), frame.image_id_norm.astype(str)))
    keys_valid = list(zip(valid.dataset.astype(str), valid.image_id_norm.astype(str)))
    if keys_frame != keys_valid:
        raise RuntimeError("06A baseline validation predictions do not align with frozen validation manifest")
    if not np.array_equal(frame.label.to_numpy(dtype=int), valid.label.to_numpy(dtype=int)):
        raise RuntimeError("06A baseline validation labels do not align")
    probs = frame[[f"prob_{c}" for c in CLASSES]].to_numpy(dtype=np.float64)
    if not np.isfinite(probs).all() or np.any(probs <= 0):
        raise RuntimeError("Invalid baseline validation probabilities")
    probs = probs / probs.sum(axis=1, keepdims=True)
    met = metrics(valid.label.to_numpy(dtype=int), probs, valid.dataset.astype(str).tolist())
    assert_baseline_metrics(met)
    return probs.astype(np.float32), met, source


def assert_baseline_metrics(observed: dict) -> None:
    tol = 1e-10
    for key in (
        "equal_domain_macro_f1",
        "equal_domain_balanced_accuracy",
        "equal_domain_melanoma_recall",
        "equal_domain_accuracy",
    ):
        if abs(observed[key] - BASELINE_METRICS[key]) > tol:
            raise RuntimeError(f"Locked 06A baseline metric changed for {key}: {observed[key]}")
    for d in DOMAINS:
        for key in ("accuracy", "macro_f1"):
            if abs(observed["per_domain"][d][key] - BASELINE_METRICS["per_domain"][d][key]) > tol:
                raise RuntimeError(f"Locked 06A baseline metric changed for {d}/{key}")


# ============================================================================
# FROZEN 06A FEATURE EXTRACTION
# ============================================================================

def validation_transform():
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )


class ImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)
        self.tf = validation_transform()

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            x = self.tf(ImageOps.exif_transpose(image).convert("RGB"))
        return x, int(row.label), D2I[str(row.dataset)], index


def image_loader(frame: pd.DataFrame, batch_size: int) -> DataLoader:
    return DataLoader(
        ImageDataset(frame),
        batch_size=batch_size,
        shuffle=False,
        num_workers=FEATURE_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )


def build_model(family: str) -> nn.Module:
    if family == "B0":
        m = models.efficientnet_b0(weights=None)
    elif family == "B3":
        m = models.efficientnet_b3(weights=None)
    else:
        raise ValueError(family)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
    return m


def extract_state(obj):
    if isinstance(obj, dict):
        for key in ("model_state_dict", "state_dict", "model", "ema_state_dict", "model_ema_state_dict"):
            value = obj.get(key)
            if isinstance(value, dict) and value and all(torch.is_tensor(v) for v in value.values()):
                return value
        if obj and all(torch.is_tensor(v) for v in obj.values()):
            return obj
    raise RuntimeError("No model state_dict found")


def normalize_state_dict(state):
    keys = list(state)
    if keys and all(str(k).startswith("module.") for k in keys):
        return {str(k)[7:]: v for k, v in state.items()}
    if keys and all(str(k).startswith("_orig_mod.") for k in keys):
        return {str(k)[10:]: v for k, v in state.items()}
    return state


def load_06a_model(path: Path, family: str, device: torch.device) -> nn.Module:
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if tuple(obj.get("class_order", ())) != CLASSES:
        raise RuntimeError(f"06A checkpoint class order mismatch: {path}")
    m = build_model(family)
    state = normalize_state_dict(extract_state(obj))
    m.load_state_dict(state, strict=True)
    m.to(device).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m


@torch.no_grad()
def forward_embedding_logits(model: nn.Module, x: torch.Tensor, device: torch.device):
    # Implementation revision v2: frozen 06A feature extraction is forced to
    # float32. The v1 preflight run aborted before adapter training because B3
    # penultimate activations became non-finite under CUDA float16 autocast.
    # This does not alter any 06A weights, data split, adapter architecture,
    # loss, epoch count, selection rule, or acceptance gate.
    x = x.to(device=device, dtype=torch.float32, non_blocking=True)
    z = model.features(x)
    z = model.avgpool(z)
    z = torch.flatten(z, 1)
    logits = model.classifier(z)
    return z, logits


@torch.no_grad()
def extract_family(
    family: str,
    names: tuple[str, ...],
    checkpoint_paths: dict[str, Path],
    train: pd.DataFrame,
    valid: pd.DataFrame,
    device: torch.device,
):
    batch_size = FEATURE_BATCH_B0 if family == "B0" else FEATURE_BATCH_B3
    models_family = [load_06a_model(checkpoint_paths[n], family, device) for n in names]
    emb_dim = models_family[0].classifier[1].in_features

    def run(frame: pd.DataFrame, split_name: str):
        loader = image_loader(frame, batch_size)
        # Zero initialization plus an explicit filled mask prevents uninitialized
        # np.empty memory from ever being mistaken for a numerical model failure.
        embeddings = np.zeros((len(frame), emb_dim), dtype=np.float32)
        probabilities = np.zeros((len(frame), len(CLASSES)), dtype=np.float32)
        filled = np.zeros(len(frame), dtype=bool)
        for x, _, _, idx in tqdm(loader, desc=f"{family} frozen features {split_name}"):
            if not torch.isfinite(x).all():
                bad = idx[~torch.isfinite(x).flatten(1).all(dim=1)].tolist()
                raise RuntimeError(f"Non-finite input tensor for {family}/{split_name}; indices={bad[:10]}")
            emb_sum = None
            prob_sum = None
            for model_name, m in zip(names, models_family):
                emb, logits = forward_embedding_logits(m, x, device)
                if not torch.isfinite(emb).all() or not torch.isfinite(logits).all():
                    bad_mask = (~torch.isfinite(emb).all(dim=1)) | (~torch.isfinite(logits).all(dim=1))
                    bad_idx = idx[bad_mask.cpu()].tolist()
                    raise RuntimeError(
                        f"Non-finite FP32 frozen output for {family}/{split_name}/{model_name}; "
                        f"indices={bad_idx[:10]}"
                    )
                prob = F.softmax(logits, dim=1)
                if not torch.isfinite(prob).all():
                    bad_idx = idx[(~torch.isfinite(prob).all(dim=1)).cpu()].tolist()
                    raise RuntimeError(
                        f"Non-finite FP32 probability for {family}/{split_name}/{model_name}; "
                        f"indices={bad_idx[:10]}"
                    )
                emb_sum = emb if emb_sum is None else emb_sum + emb
                prob_sum = prob if prob_sum is None else prob_sum + prob
            emb_mean = emb_sum / len(models_family)
            prob_mean = prob_sum / len(models_family)
            ii = idx.numpy()
            embeddings[ii] = emb_mean.cpu().numpy()
            probabilities[ii] = prob_mean.cpu().numpy()
            filled[ii] = True
        if not filled.all():
            missing = np.flatnonzero(~filled)
            raise RuntimeError(
                f"Frozen feature extraction did not fill every row for {family}/{split_name}; "
                f"missing_indices={missing[:10].tolist()} count={len(missing)}"
            )
        if not np.isfinite(embeddings).all() or not np.isfinite(probabilities).all():
            raise RuntimeError(f"Non-finite FP32 frozen features for {family}/{split_name}")
        row_sums = probabilities.sum(axis=1, keepdims=True)
        if not np.isfinite(row_sums).all() or np.any(row_sums <= 0):
            raise RuntimeError(f"Invalid FP32 probability row sums for {family}/{split_name}")
        probabilities /= row_sums
        return embeddings, probabilities

    tr_emb, tr_prob = run(train, "TRAIN")
    va_emb, va_prob = run(valid, "VALIDATION")
    del models_family
    torch.cuda.empty_cache()
    return tr_emb, tr_prob, va_emb, va_prob


def make_stats(p_b0: np.ndarray, p_b3: np.ndarray) -> np.ndarray:
    eps = 1e-7
    p = 0.5 * p_b0 + 0.5 * p_b3
    p /= p.sum(axis=1, keepdims=True)
    def entropy(q):
        return -(q * np.log(np.clip(q, eps, 1.0))).sum(axis=1, keepdims=True)
    stats = np.concatenate(
        [
            p_b0,
            p_b3,
            p,
            entropy(p_b0),
            entropy(p_b3),
            entropy(p),
            np.abs(p_b0 - p_b3),
            p_b0.max(axis=1, keepdims=True),
            p_b3.max(axis=1, keepdims=True),
            p.max(axis=1, keepdims=True),
        ],
        axis=1,
    )
    return stats.astype(np.float32)


# ============================================================================
# RESIDUAL ADAPTER
# ============================================================================

class FeatureDataset(Dataset):
    def __init__(
        self,
        b0: np.ndarray,
        b3: np.ndarray,
        stats: np.ndarray,
        base_prob: np.ndarray,
        labels: np.ndarray,
        domains: np.ndarray,
        sample_weights: np.ndarray,
    ):
        self.b0 = torch.from_numpy(b0)
        self.b3 = torch.from_numpy(b3)
        self.stats = torch.from_numpy(stats)
        self.base_prob = torch.from_numpy(base_prob)
        self.labels = torch.from_numpy(labels.astype(np.int64))
        self.domains = torch.from_numpy(domains.astype(np.int64))
        self.sample_weights = torch.from_numpy(sample_weights.astype(np.float32))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return (
            self.b0[i],
            self.b3[i],
            self.stats[i],
            self.base_prob[i],
            self.labels[i],
            self.domains[i],
            self.sample_weights[i],
        )


class SoftRoutedResidualAdapter(nn.Module):
    def __init__(self, b0_dim: int, b3_dim: int, stat_dim: int):
        super().__init__()
        self.b0_proj = nn.Sequential(nn.LayerNorm(b0_dim), nn.Linear(b0_dim, 64), nn.GELU())
        self.b3_proj = nn.Sequential(nn.LayerNorm(b3_dim), nn.Linear(b3_dim, 64), nn.GELU())
        self.stat_proj = nn.Sequential(nn.LayerNorm(stat_dim), nn.Linear(stat_dim, 32), nn.GELU())
        self.trunk = nn.Sequential(nn.Linear(160, 128), nn.GELU(), nn.Dropout(DROPOUT))
        self.router = nn.Linear(128, len(DOMAINS))
        self.experts = nn.ModuleList([nn.Linear(128, len(CLASSES)) for _ in DOMAINS])
        # Exact baseline at initialization: all residual experts start at zero.
        for expert in self.experts:
            nn.init.zeros_(expert.weight)
            nn.init.zeros_(expert.bias)

    def forward(self, b0, b3, stats):
        h = torch.cat([self.b0_proj(b0), self.b3_proj(b3), self.stat_proj(stats)], dim=1)
        h = self.trunk(h)
        router_logits = self.router(h)
        gate = F.softmax(router_logits / ROUTER_TEMPERATURE, dim=1)
        raw = torch.stack([expert(h) for expert in self.experts], dim=1)
        mixed_raw = (gate.unsqueeze(-1) * raw).sum(dim=1)
        correction = RESIDUAL_LOGIT_CAP * torch.tanh(mixed_raw)
        return correction, router_logits, gate


def domain_sample_weights(train: pd.DataFrame) -> tuple[np.ndarray, dict]:
    counts = train.dataset.value_counts().to_dict()
    max_count = max(counts.values())
    raw = {d: math.sqrt(max_count / counts[d]) for d in DOMAINS}
    weights = np.asarray([raw[str(d)] for d in train.dataset], dtype=np.float64)
    weights /= weights.mean()
    normalized = {d: float(raw[d] / weights.mean()) for d in DOMAINS}
    # normalized dict above is only descriptive; actual array has mean 1.
    return weights.astype(np.float32), {
        "counts": {d: int(counts[d]) for d in DOMAINS},
        "formula": "sqrt(max_domain_count / domain_count), then normalize sample-weight mean to 1",
        "raw_domain_weights": {d: float(raw[d]) for d in DOMAINS},
        "sample_weight_min": float(weights.min()),
        "sample_weight_max": float(weights.max()),
        "sample_weight_mean": float(weights.mean()),
    }


def candidate_probabilities(base_prob: torch.Tensor, correction: torch.Tensor) -> torch.Tensor:
    logits = torch.log(base_prob.clamp_min(1e-7)) + correction
    return F.softmax(logits, dim=1)


def weighted_mean(values: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    return (values * weights).sum() / weights.sum().clamp_min(1e-8)


@torch.no_grad()
def adapter_outputs(
    adapter: nn.Module,
    b0: np.ndarray,
    b3: np.ndarray,
    stats: np.ndarray,
    device: torch.device,
    batch_size: int = 1024,
):
    adapter.eval()
    corrections = []
    gates = []
    routers = []
    for start in range(0, len(b0), batch_size):
        stop = min(start + batch_size, len(b0))
        tb0 = torch.from_numpy(b0[start:stop]).to(device)
        tb3 = torch.from_numpy(b3[start:stop]).to(device)
        ts = torch.from_numpy(stats[start:stop]).to(device)
        corr, router_logits, gate = adapter(tb0, tb3, ts)
        corrections.append(corr.cpu().numpy())
        gates.append(gate.cpu().numpy())
        routers.append(router_logits.cpu().numpy())
    return np.concatenate(corrections), np.concatenate(gates), np.concatenate(routers)


def train_adapter_seed(
    seed: int,
    train_features: dict,
    valid_features: dict,
    valid_base_tta7: np.ndarray,
    train_frame: pd.DataFrame,
    valid_frame: pd.DataFrame,
    sample_weights: np.ndarray,
    device: torch.device,
):
    seed_all(seed)
    adapter = SoftRoutedResidualAdapter(
        train_features["b0"].shape[1],
        train_features["b3"].shape[1],
        train_features["stats"].shape[1],
    ).to(device)
    optimizer = torch.optim.AdamW(adapter.parameters(), lr=ADAPTER_LR, weight_decay=ADAPTER_WEIGHT_DECAY)

    ds = FeatureDataset(
        train_features["b0"],
        train_features["b3"],
        train_features["stats"],
        train_features["base_prob"],
        train_frame.label.to_numpy(dtype=np.int64),
        train_frame.dataset.map(D2I).to_numpy(dtype=np.int64),
        sample_weights,
    )
    generator = torch.Generator()
    generator.manual_seed(seed)
    loader = DataLoader(ds, batch_size=ADAPTER_BATCH, shuffle=True, generator=generator, num_workers=0)

    history = []
    states = {0: {k: v.detach().cpu().clone() for k, v in adapter.state_dict().items()}}
    val_corr_by_epoch = {0: np.zeros((len(valid_frame), len(CLASSES)), dtype=np.float32)}
    val_gate_by_epoch = {0: np.full((len(valid_frame), len(DOMAINS)), 1.0 / len(DOMAINS), dtype=np.float32)}

    for epoch in range(1, ADAPTER_EPOCHS + 1):
        adapter.train()
        totals = {"loss": 0.0, "classification": 0.0, "source_anchor": 0.0, "router": 0.0, "energy": 0.0}
        seen = 0
        for b0, b3, stats, base_prob, labels, domain_idx, weights in loader:
            b0 = b0.to(device)
            b3 = b3.to(device)
            stats = stats.to(device)
            base_prob = base_prob.to(device)
            labels = labels.to(device)
            domain_idx = domain_idx.to(device)
            weights = weights.to(device)

            correction, router_logits, _ = adapter(b0, b3, stats)
            candidate_logits = torch.log(base_prob.clamp_min(1e-7)) + correction
            per_ce = F.cross_entropy(
                candidate_logits,
                labels,
                reduction="none",
                label_smoothing=LABEL_SMOOTHING,
            )
            classification = weighted_mean(per_ce, weights)

            candidate_log_prob = F.log_softmax(candidate_logits, dim=1)
            isic = domain_idx.eq(D2I["ISIC2018_HAM10000"])
            if bool(isic.any()):
                teacher = base_prob[isic].detach().clamp_min(1e-7)
                anchor_per = F.kl_div(candidate_log_prob[isic], teacher, reduction="none").sum(dim=1)
                source_anchor = anchor_per.mean()
            else:
                source_anchor = candidate_logits.new_tensor(0.0)

            router_per = F.cross_entropy(router_logits, domain_idx, reduction="none")
            router_loss = weighted_mean(router_per, weights)
            residual_energy = correction.pow(2).mean()

            loss = (
                classification
                + SOURCE_ANCHOR_WEIGHT * source_anchor
                + ROUTER_AUX_WEIGHT * router_loss
                + RESIDUAL_ENERGY_WEIGHT * residual_energy
            )
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite adapter training loss")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 5.0)
            optimizer.step()

            n = len(labels)
            seen += n
            totals["loss"] += float(loss.detach().cpu()) * n
            totals["classification"] += float(classification.detach().cpu()) * n
            totals["source_anchor"] += float(source_anchor.detach().cpu()) * n
            totals["router"] += float(router_loss.detach().cpu()) * n
            totals["energy"] += float(residual_energy.detach().cpu()) * n

        if seen != len(train_frame):
            raise RuntimeError(f"Each TRAIN image must be seen exactly once per adapter epoch: {seen} vs {len(train_frame)}")

        correction, gate, router_logits = adapter_outputs(
            adapter,
            valid_features["b0"],
            valid_features["b3"],
            valid_features["stats"],
            device,
        )
        val_corr_by_epoch[epoch] = correction.astype(np.float32)
        val_gate_by_epoch[epoch] = gate.astype(np.float32)
        states[epoch] = {k: v.detach().cpu().clone() for k, v in adapter.state_dict().items()}

        candidate = softmax_numpy(np.log(np.clip(valid_base_tta7, 1e-7, 1.0)) + correction)
        met = metrics(valid_frame.label.to_numpy(dtype=int), candidate, valid_frame.dataset.astype(str).tolist())
        row = {
            "seed": seed,
            "epoch": epoch,
            **{f"train_{k}": totals[k] / seen for k in totals},
            **{k: v for k, v in met.items() if k != "per_domain"},
        }
        for d in DOMAINS:
            for key, value in met["per_domain"][d].items():
                if key != "n":
                    row[f"{d}_{key}"] = value
        history.append(row)
        print("[ADAPTER]", json.dumps(row))

    pd.DataFrame(history).to_csv(TAB / f"adapter_seed_{seed}_training_history.csv", index=False)
    return {
        "seed": seed,
        "states": states,
        "val_corr": val_corr_by_epoch,
        "val_gate": val_gate_by_epoch,
    }


# ============================================================================
# METRICS / SELECTION
# ============================================================================

def softmax_numpy(logits: np.ndarray) -> np.ndarray:
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def metrics(y, p, domains) -> dict:
    y = np.asarray(y)
    p = np.asarray(p)
    pred = p.argmax(axis=1)
    dom = np.asarray(domains, dtype=object)
    per = {}
    for d in DOMAINS:
        mask = dom == d
        yy = y[mask]
        pp = p[mask]
        pr = pred[mask]
        labels = [C2I[c] for c in PRESENT[d]]
        recalls = [
            recall_score(yy, pr, labels=[i], average="macro", zero_division=0)
            for i in labels
        ]
        aucs = []
        for i in labels:
            binary = (yy == i).astype(int)
            if len(np.unique(binary)) == 2:
                aucs.append(roc_auc_score(binary, pp[:, i]))
        per[d] = {
            "n": int(mask.sum()),
            "accuracy": float(accuracy_score(yy, pr)),
            "macro_f1": float(f1_score(yy, pr, labels=labels, average="macro", zero_division=0)),
            "balanced_accuracy": float(np.mean(recalls)),
            "melanoma_recall": float(recall_score(yy, pr, labels=[0], average="macro", zero_division=0)),
            "macro_auc": float(np.mean(aucs)),
        }
    return {
        "per_domain": per,
        "equal_domain_macro_f1": float(np.mean([per[d]["macro_f1"] for d in DOMAINS])),
        "equal_domain_balanced_accuracy": float(np.mean([per[d]["balanced_accuracy"] for d in DOMAINS])),
        "equal_domain_melanoma_recall": float(np.mean([per[d]["melanoma_recall"] for d in DOMAINS])),
        "equal_domain_accuracy": float(np.mean([per[d]["accuracy"] for d in DOMAINS])),
        "equal_domain_macro_auc": float(np.mean([per[d]["macro_auc"] for d in DOMAINS])),
    }


def delta(candidate: dict, baseline: dict, domain: str | None, metric: str) -> float:
    if domain is None:
        return float(candidate[metric] - baseline[metric])
    return float(candidate["per_domain"][domain][metric] - baseline["per_domain"][domain][metric])


def eligibility_checks(candidate: dict, baseline: dict) -> dict[str, bool]:
    return {
        "equal_domain_balanced_accuracy": delta(candidate, baseline, None, "equal_domain_balanced_accuracy") >= ELIGIBILITY["equal_domain_balanced_accuracy_min_delta"],
        "equal_domain_melanoma_recall": delta(candidate, baseline, None, "equal_domain_melanoma_recall") >= ELIGIBILITY["equal_domain_melanoma_recall_min_delta"],
        "ISIC_macro_f1": delta(candidate, baseline, "ISIC2018_HAM10000", "macro_f1") >= ELIGIBILITY["ISIC2018_HAM10000_macro_f1_min_delta"],
        "ISIC_accuracy": delta(candidate, baseline, "ISIC2018_HAM10000", "accuracy") >= ELIGIBILITY["ISIC2018_HAM10000_accuracy_min_delta"],
        "BCN_macro_f1": delta(candidate, baseline, "BCN20000", "macro_f1") >= ELIGIBILITY["BCN20000_macro_f1_min_delta"],
        "Derm7pt_macro_f1": delta(candidate, baseline, "Derm7pt", "macro_f1") >= ELIGIBILITY["Derm7pt_macro_f1_min_delta"],
        "PAD_macro_f1": delta(candidate, baseline, "PAD_UFES_20", "macro_f1") >= ELIGIBILITY["PAD_UFES_20_macro_f1_min_delta"],
        "PAD_accuracy": delta(candidate, baseline, "PAD_UFES_20", "accuracy") >= ELIGIBILITY["PAD_UFES_20_accuracy_min_delta"],
    }


def acceptance_checks(candidate: dict, baseline: dict) -> dict[str, bool]:
    return {
        "equal_domain_macro_f1_improvement": delta(candidate, baseline, None, "equal_domain_macro_f1") >= ACCEPTANCE["equal_domain_macro_f1_min_improvement"],
        "equal_domain_balanced_accuracy": delta(candidate, baseline, None, "equal_domain_balanced_accuracy") >= ACCEPTANCE["equal_domain_balanced_accuracy_min_delta"],
        "equal_domain_melanoma_recall": delta(candidate, baseline, None, "equal_domain_melanoma_recall") >= ACCEPTANCE["equal_domain_melanoma_recall_min_delta"],
        "ISIC_macro_f1": delta(candidate, baseline, "ISIC2018_HAM10000", "macro_f1") >= ACCEPTANCE["ISIC2018_HAM10000_macro_f1_min_delta"],
        "ISIC_accuracy": delta(candidate, baseline, "ISIC2018_HAM10000", "accuracy") >= ACCEPTANCE["ISIC2018_HAM10000_accuracy_min_delta"],
        "BCN_macro_f1": delta(candidate, baseline, "BCN20000", "macro_f1") >= ACCEPTANCE["BCN20000_macro_f1_min_delta"],
        "Derm7pt_macro_f1": delta(candidate, baseline, "Derm7pt", "macro_f1") >= ACCEPTANCE["Derm7pt_macro_f1_min_delta"],
        "PAD_macro_f1": delta(candidate, baseline, "PAD_UFES_20", "macro_f1") >= ACCEPTANCE["PAD_UFES_20_macro_f1_min_delta"],
        "PAD_accuracy": delta(candidate, baseline, "PAD_UFES_20", "accuracy") >= ACCEPTANCE["PAD_UFES_20_accuracy_min_delta"],
    }


def selection_key(metrics_dict: dict, epoch: int):
    return (
        metrics_dict["equal_domain_macro_f1"],
        metrics_dict["equal_domain_balanced_accuracy"],
        metrics_dict["equal_domain_melanoma_recall"],
        metrics_dict["per_domain"]["ISIC2018_HAM10000"]["macro_f1"],
        -epoch,
    )


def save_predictions(path: Path, frame: pd.DataFrame, probs: np.ndarray, model_name: str):
    out = frame[["dataset", "image_id_norm", "group_id", "class_name", "label"]].copy()
    pred = probs.argmax(axis=1)
    out["model_name"] = model_name
    out["predicted_label"] = pred
    out["predicted_class"] = [CLASSES[i] for i in pred]
    out["correct"] = (pred == out.label.to_numpy()).astype(int)
    for i, c in enumerate(CLASSES):
        out[f"prob_{c}"] = probs[:, i]
    out.to_csv(path, index=False)


# ============================================================================
# PACKAGING
# ============================================================================

def make_material_index():
    rows = []
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and "verified_06a_checkpoints" not in p.parts:
            rows.append(
                {
                    "file": str(p.relative_to(OUT)),
                    "sha256": sha_file(p),
                    "bytes": p.stat().st_size,
                    "checkpoint": p.suffix.lower() in {".pth", ".pt", ".ckpt"},
                }
            )
    pd.DataFrame(rows).to_csv(OUT / "paper_material_index.csv", index=False)


def make_archives():
    results_zip = WORKING_ROOT / "PAPEREXP06D_Frozen_Backbone_Soft_Routed_Residual_Adaptation_Results.zip"
    recovery_zip = WORKING_ROOT / "PAPEREXP06D_Frozen_Backbone_Soft_Routed_Residual_Adaptation_Recovery.zip"
    with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob("*")):
            if not p.is_file() or "verified_06a_checkpoints" in p.parts:
                continue
            if p.suffix.lower() in {".pth", ".pt", ".ckpt"}:
                continue
            z.write(p, arcname=str(p.relative_to(OUT)))
    with zipfile.ZipFile(recovery_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob("*")):
            if not p.is_file() or "verified_06a_checkpoints" in p.parts:
                continue
            z.write(p, arcname=str(p.relative_to(OUT)))
    return results_zip, recovery_zip


# ============================================================================
# MAIN
# ============================================================================

def main():
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU accelerator")
    device = torch.device("cuda:0")
    print("[GPU]", torch.cuda.get_device_name(0))
    print("[CODE VERSION]", CODE_VERSION)
    print("[STATUS] DEVELOPMENT ONLY. FROZEN TESTS AND DERM12345 REMAIN UNOPENED.")

    derm_boundary = assert_derm12345_boundary()
    train, valid, leakage, manifest_sources = load_manifests()
    train, valid = resolve_all(train, valid)
    checkpoint_paths, checkpoint_locations = find_06a_checkpoints()
    baseline_tta7, baseline_metrics, baseline_prediction_source = load_baseline_validation(valid)

    sample_weights, domain_weight_info = domain_sample_weights(train)
    preflight = {
        "experiment": EXPERIMENT,
        "code_version": CODE_VERSION,
        "train_images": len(train),
        "validation_images": len(valid),
        "train_manifest_sha256": TRAIN_SHA,
        "validation_manifest_sha256": VALID_SHA,
        "manifest_sources": manifest_sources,
        "leakage": leakage,
        "baseline_validation_prediction_sha256": BASELINE_VALIDATION_SHA,
        "baseline_validation_prediction_source": baseline_prediction_source,
        "baseline_checkpoint_sha256": BASELINE_CKPT_HASHES,
        "baseline_checkpoint_locations": checkpoint_locations,
        "frozen_source_retention_test_read": False,
        "frozen_external_test_read": False,
        "DERM12345_read": False,
        "DERM12345_boundary": derm_boundary,
        "domain_weighting": domain_weight_info,
        "no_backbone_training": True,
        "no_classifier_training": True,
        "frozen_feature_extraction_precision": "float32_no_autocast",
        "implementation_revision": "v2 fixes v1 pre-training FP16 B3 feature overflow; scientific protocol unchanged",
        "weight_search": False,
        "TTA_search": False,
        "threshold_search": False,
        "calibration": False,
    }
    json_dump(preflight, AUDIT / "preflight.json")

    # Exact frozen 06A features. No augmentation and no gradient into 06A.
    tr_b0, tr_p0, va_b0, va_p0 = extract_family("B0", B0_NAMES, checkpoint_paths, train, valid, device)
    tr_b3, tr_p3, va_b3, va_p3 = extract_family("B3", B3_NAMES, checkpoint_paths, train, valid, device)
    train_base_sv = 0.5 * tr_p0 + 0.5 * tr_p3
    valid_base_sv = 0.5 * va_p0 + 0.5 * va_p3
    train_base_sv /= train_base_sv.sum(axis=1, keepdims=True)
    valid_base_sv /= valid_base_sv.sum(axis=1, keepdims=True)
    train_features = {
        "b0": tr_b0,
        "b3": tr_b3,
        "stats": make_stats(tr_p0, tr_p3),
        "base_prob": train_base_sv.astype(np.float32),
    }
    valid_features = {
        "b0": va_b0,
        "b3": va_b3,
        "stats": make_stats(va_p0, va_p3),
        "base_prob": valid_base_sv.astype(np.float32),
    }

    # Train three adapter seeds. The final system uses the mean residual at one
    # COMMON epoch selected at the system level, never per-seed checkpoint picks.
    runs = []
    for seed in ADAPTER_SEEDS:
        runs.append(
            train_adapter_seed(
                seed,
                train_features,
                valid_features,
                baseline_tta7,
                train,
                valid,
                sample_weights,
                device,
            )
        )

    epoch_rows = []
    epoch_metrics = {}
    epoch_probs = {}
    epoch_gates = {}
    labels = valid.label.to_numpy(dtype=int)
    domains = valid.dataset.astype(str).tolist()

    for epoch in range(0, ADAPTER_EPOCHS + 1):
        mean_correction = np.mean([run["val_corr"][epoch] for run in runs], axis=0)
        mean_gate = np.mean([run["val_gate"][epoch] for run in runs], axis=0)
        probs = softmax_numpy(np.log(np.clip(baseline_tta7, 1e-7, 1.0)) + mean_correction)
        met = metrics(labels, probs, domains)
        checks = eligibility_checks(met, baseline_metrics)
        eligible = all(checks.values())
        epoch_metrics[epoch] = met
        epoch_probs[epoch] = probs
        epoch_gates[epoch] = mean_gate
        row = {
            "epoch": epoch,
            "eligible": eligible,
            **{f"elig_{k}": v for k, v in checks.items()},
            "equal_domain_macro_f1": met["equal_domain_macro_f1"],
            "equal_domain_balanced_accuracy": met["equal_domain_balanced_accuracy"],
            "equal_domain_melanoma_recall": met["equal_domain_melanoma_recall"],
            "equal_domain_accuracy": met["equal_domain_accuracy"],
        }
        for d in DOMAINS:
            for key in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall"):
                row[f"{d}_{key}"] = met["per_domain"][d][key]
        epoch_rows.append(row)
        print("[SYSTEM EPOCH]", json.dumps(row))

    pd.DataFrame(epoch_rows).to_csv(TAB / "Table3_system_level_epoch_selection.csv", index=False)

    eligible_epochs = [e for e in range(ADAPTER_EPOCHS + 1) if all(eligibility_checks(epoch_metrics[e], baseline_metrics).values())]
    if not eligible_epochs:
        raise RuntimeError("Internal error: epoch 0 baseline must always be selection-eligible")
    selected_epoch = max(eligible_epochs, key=lambda e: selection_key(epoch_metrics[e], e))
    selected_metrics = epoch_metrics[selected_epoch]
    selected_probs = epoch_probs[selected_epoch]
    selected_gates = epoch_gates[selected_epoch]
    gate_checks = acceptance_checks(selected_metrics, baseline_metrics)
    gate_passed = all(gate_checks.values())

    # Save the three adapter states at the one common selected system epoch.
    adapter_hashes = {}
    for run in runs:
        seed = run["seed"]
        path = CKPT / f"adapter_seed_{seed}" / "selected.pth"
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "experiment": EXPERIMENT,
            "title": TITLE,
            "code_version": CODE_VERSION,
            "seed": seed,
            "selected_common_epoch": selected_epoch,
            "class_order": list(CLASSES),
            "domain_order": list(DOMAINS),
            "adapter_state_dict": run["states"][selected_epoch],
            "architecture": {
                "B0_projection": 64,
                "B3_projection": 64,
                "stats_projection": 32,
                "trunk": 128,
                "experts": len(DOMAINS),
                "residual_logit_cap": RESIDUAL_LOGIT_CAP,
                "router_temperature": ROUTER_TEMPERATURE,
            },
            "parent_06A_checkpoint_sha256": BASELINE_CKPT_HASHES,
            "train_manifest_sha256": TRAIN_SHA,
            "validation_manifest_sha256": VALID_SHA,
            "baseline_validation_prediction_sha256": BASELINE_VALIDATION_SHA,
            "selected_system_validation_metrics": selected_metrics,
        }
        torch.save(payload, path)
        adapter_hashes[str(seed)] = sha_file(path)

    save_predictions(
        PRED / "BASELINE_06A_fixed_50B0_50B3_tta7_validation.csv",
        valid,
        baseline_tta7,
        "PAPER-EXP-06A",
    )
    save_predictions(
        PRED / "PAPEREXP06D_soft_routed_residual_tta7_validation.csv",
        valid,
        selected_probs,
        "PAPER-EXP-06D",
    )

    # Tables.
    rows = []
    for system_name, met in (("PAPER-EXP-06A", baseline_metrics), ("PAPER-EXP-06D", selected_metrics)):
        for d in DOMAINS:
            rows.append({"system": system_name, "dataset": d, **met["per_domain"][d]})
        rows.append(
            {
                "system": system_name,
                "dataset": "EQUAL_DOMAIN_MEAN",
                "n": None,
                "accuracy": met["equal_domain_accuracy"],
                "macro_f1": met["equal_domain_macro_f1"],
                "balanced_accuracy": met["equal_domain_balanced_accuracy"],
                "melanoma_recall": met["equal_domain_melanoma_recall"],
                "macro_auc": met["equal_domain_macro_auc"],
            }
        )
    pd.DataFrame(rows).to_csv(TAB / "Table1_baseline_vs_PAPEREXP06D_validation.csv", index=False)

    delta_rows = []
    for d in DOMAINS:
        delta_rows.append(
            {
                "dataset": d,
                **{
                    f"delta_{k}": selected_metrics["per_domain"][d][k] - baseline_metrics["per_domain"][d][k]
                    for k in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc")
                },
            }
        )
    delta_rows.append(
        {
            "dataset": "EQUAL_DOMAIN_MEAN",
            "delta_accuracy": selected_metrics["equal_domain_accuracy"] - baseline_metrics["equal_domain_accuracy"],
            "delta_macro_f1": selected_metrics["equal_domain_macro_f1"] - baseline_metrics["equal_domain_macro_f1"],
            "delta_balanced_accuracy": selected_metrics["equal_domain_balanced_accuracy"] - baseline_metrics["equal_domain_balanced_accuracy"],
            "delta_melanoma_recall": selected_metrics["equal_domain_melanoma_recall"] - baseline_metrics["equal_domain_melanoma_recall"],
            "delta_macro_auc": selected_metrics["equal_domain_macro_auc"] - baseline_metrics["equal_domain_macro_auc"],
        }
    )
    pd.DataFrame(delta_rows).to_csv(TAB / "Table2_PAPEREXP06D_minus_baseline_validation_deltas.csv", index=False)

    router_rows = []
    domain_array = valid.dataset.astype(str).to_numpy()
    for actual_domain in DOMAINS:
        mask = domain_array == actual_domain
        avg_gate = selected_gates[mask].mean(axis=0)
        route_pred = selected_gates[mask].argmax(axis=1)
        router_rows.append(
            {
                "actual_domain": actual_domain,
                "n": int(mask.sum()),
                "router_top1_accuracy_within_domain": float(np.mean(route_pred == D2I[actual_domain])),
                **{f"mean_gate_to_{DOMAINS[j]}": float(avg_gate[j]) for j in range(len(DOMAINS))},
            }
        )
    pd.DataFrame(router_rows).to_csv(TAB / "Table4_router_behavior_selected_epoch.csv", index=False)

    protocol = {
        "experiment": EXPERIMENT,
        "title": TITLE,
        "code_version": CODE_VERSION,
        "scientific_role": "Final planned generalized-model development experiment before lock/frozen evaluation.",
        "hypothesis": "Controlled image-conditioned specialization added on top of a completely frozen PAPER-EXP-06A ensemble can improve the four-domain validation frontier without the representation drift observed in PAPER-EXP-06B and PAPER-EXP-06C.",
        "parent": {
            "model": "PAPER-EXP-06A",
            "checkpoint_sha256": BASELINE_CKPT_HASHES,
            "baseline_validation_prediction_sha256": BASELINE_VALIDATION_SHA,
            "all_06A_parameters_frozen": True,
        },
        "development_data": {
            "train_images": 16220,
            "validation_images": 3895,
            "train_sha256": TRAIN_SHA,
            "validation_sha256": VALID_SHA,
            "domains": list(DOMAINS),
            "classes": list(CLASSES),
            "DERM12345_used": False,
            "frozen_source_retention_test_used": False,
            "frozen_external_test_used": False,
        },
        "frozen_feature_path": {
            "view": "deterministic 300x300 original view only",
            "B0_embedding": "mean penultimate embedding across exact three frozen 06A B0 checkpoints",
            "B3_embedding": "mean penultimate embedding across exact three frozen 06A B3 checkpoints",
            "probability_stats": "B0/B3/fused original-view probabilities, entropies, disagreement, confidences",
        },
        "adapter": {
            "B0_projection": 64,
            "B3_projection": 64,
            "stats_projection": 32,
            "shared_trunk": 128,
            "soft_router_experts": 4,
            "router_supervision": "TRAIN domain identity only; dataset identity is never passed at validation/inference",
            "residual_logit_cap": RESIDUAL_LOGIT_CAP,
            "residual_experts_zero_initialized": True,
            "final_formula": "softmax(log(PAPER-EXP-06A TTA7 probability) + mean_of_three_adapter_residuals)",
        },
        "training": {
            "adapter_seeds": list(ADAPTER_SEEDS),
            "epochs": ADAPTER_EPOCHS,
            "batch_size": ADAPTER_BATCH,
            "optimizer": "AdamW",
            "lr": ADAPTER_LR,
            "weight_decay": ADAPTER_WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "each_train_image_exactly_once_per_epoch": True,
            "domain_weighting": "sqrt(max_domain_count/domain_count), derived only from TRAIN counts",
            "source_ISIC_KL_anchor_weight": SOURCE_ANCHOR_WEIGHT,
            "router_auxiliary_weight": ROUTER_AUX_WEIGHT,
            "residual_energy_weight": RESIDUAL_ENERGY_WEIGHT,
            "augmentation": "none during adapter training; frozen deterministic features",
        },
        "selection": {
            "unit": "actual final three-adapter residual ensemble at one COMMON epoch",
            "epochs_considered": list(range(0, ADAPTER_EPOCHS + 1)),
            "epoch_0": "exact PAPER-EXP-06A baseline",
            "eligibility": ELIGIBILITY,
            "ranking": [
                "equal-domain validation Macro-F1",
                "equal-domain validation balanced accuracy",
                "equal-domain validation melanoma recall",
                "ISIC validation Macro-F1",
                "earlier common epoch",
            ],
            "no_per_seed_validation_checkpoint_selection": True,
        },
        "acceptance_gate": ACCEPTANCE,
        "hard_rules": [
            "PAPER-EXP-06A backbone and classifier weights never change.",
            "Do not use PAPER-EXP-06B or PAPER-EXP-06C checkpoints as model sources.",
            "Do not read frozen source-retention TEST during 06D.",
            "Do not read frozen external TEST during 06D.",
            "Do not read DERM12345 data, labels, predictions, or outcomes during 06D.",
            "No adapter architecture, LR, epoch count, residual cap, loss weight, fusion weight, TTA, threshold, or calibration search.",
            "If the predeclared acceptance gate fails, retain PAPER-EXP-06A and stop generalized-model development.",
        ],
    }
    json_dump(protocol, OUT / "paper_exp06d_protocol.json")

    decision = {
        "experiment": EXPERIMENT,
        "code_version": CODE_VERSION,
        "status": "COMPLETE_DEVELOPMENT_ONLY",
        "selected_common_epoch": selected_epoch,
        "adapter_checkpoint_sha256": adapter_hashes,
        "parent_06A_checkpoint_sha256": BASELINE_CKPT_HASHES,
        "baseline_validation": baseline_metrics,
        "candidate_validation": selected_metrics,
        "acceptance_gate": {
            "passed": gate_passed,
            "checks": gate_checks,
            "thresholds": ACCEPTANCE,
        },
        "decision": "ACCEPT_PAPER_EXP_06D" if gate_passed else "REJECT_PAPER_EXP_06D_KEEP_PAPER_EXP_06A",
        "frozen_source_retention_test_read": False,
        "frozen_external_test_read": False,
        "DERM12345_read": False,
        "next_if_accepted": "Lock PAPER-EXP-06D generalized system, then run once-only frozen evaluations under a new experiment number.",
        "next_if_rejected": "Lock PAPER-EXP-06A as final generalized model; do not run further generalized-model development; then run once-only frozen evaluations under a new experiment number.",
    }
    json_dump(decision, OUT / "PAPEREXP06D_FINAL_DECISION.json")

    try:
        shutil.copy2(Path(__file__), OUT / "paper_exp06d_code_snapshot.py")
    except Exception:
        pass

    make_material_index()
    results_zip, recovery_zip = make_archives()
    print("\nPAPER-EXP-06D COMPLETE")
    print("SELECTED COMMON EPOCH:", selected_epoch)
    print("ACCEPTANCE GATE PASSED:", gate_passed)
    print(json.dumps(selected_metrics, indent=2))
    print("RESULTS ZIP:", results_zip, sha_file(results_zip))
    print("RECOVERY ZIP:", recovery_zip, sha_file(recovery_zip))
    print("FROZEN TESTS READ:", False)
    print("DERM12345 READ:", False)


if __name__ == "__main__":
    main()


[GPU] Tesla T4
[CODE VERSION] PAPEREXP06D_FROZEN_06A_SOFT_ROUTED_RESIDUAL_2026-08-30_v2_FP32_FEATURE_EXTRACTION_FIX
[STATUS] DEVELOPMENT ONLY. FROZEN TESTS AND DERM12345 REMAIN UNOPENED.
[DERM12345 BOUNDARY] inert frozen holdout manifest(s) present but unused: 2
[INDEX] ISIC2018_HAM10000 20030
[INDEX] BCN20000 18946
[INDEX] Derm7pt 2013
[INDEX] PAD_UFES_20 2298


B0 frozen features TRAIN:   0%|          | 0/507 [00:00<?, ?it/s]

B0 frozen features VALIDATION:   0%|          | 0/122 [00:00<?, ?it/s]

B3 frozen features TRAIN:   0%|          | 0/1014 [00:00<?, ?it/s]

B3 frozen features VALIDATION:   0%|          | 0/244 [00:00<?, ?it/s]

[ADAPTER] {"seed": 20260829, "epoch": 1, "train_loss": 0.5180505410810581, "train_classification": 0.4302775507937524, "train_source_anchor": 0.00165036748698287, "train_router": 0.8605358800817506, "train_energy": 0.006904030520558738, "equal_domain_macro_f1": 0.7569970500809745, "equal_domain_balanced_accuracy": 0.7628821188399659, "equal_domain_melanoma_recall": 0.6603477961432507, "equal_domain_accuracy": 0.8010944954534656, "equal_domain_macro_auc": 0.9482009746852081, "ISIC2018_HAM10000_accuracy": 0.8578431372549019, "ISIC2018_HAM10000_macro_f1": 0.7624335868054561, "ISIC2018_HAM10000_balanced_accuracy": 0.7522966525576675, "ISIC2018_HAM10000_melanoma_recall": 0.5833333333333334, "ISIC2018_HAM10000_macro_auc": 0.9607192508032346, "BCN20000_accuracy": 0.7142857142857143, "BCN20000_macro_f1": 0.6570026144449965, "BCN20000_balanced_accuracy": 0.6628533902595762, "BCN20000_melanoma_recall": 0.6330578512396694, "BCN20000_macro_auc": 0.9084023180470219, "Derm7pt_accuracy": 0.8169014084